### Testing of the Model

In [ ]:
import torch
from model import SimpleCNN

# 1. Initialize our model
model = SimpleCNN(num_classes=8)

# 2. Create a "dummy" batch of images (Batch Size=1, Channels=3, Height=224, Width=224)
# We fill it with random numbers to simulate image pixels
dummy_images = torch.randn(1, 3, 224, 224)

# 3. Pass it through the model (this automatically calls the `forward` function)
output = model(dummy_images)

print("Model Output Shape:", output.shape)
print("Raw Scores for the 8 classes:\n", output.detach().numpy())

Model Output Shape: torch.Size([1, 8])
Raw Scores for the 8 classes:
 [[ 0.00190711 -0.13638073  0.05758644 -0.06877775 -0.14154379  0.07052074
  -0.06882942  0.11229552]]


### Creating a Log Book to keep history of model accuracy

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import csv
import os
from datetime import datetime

In [2]:
def log_experiment(model_name, epochs, batch_size, learning_rate, 
                   train_acc, val_acc, notes, log_file='./modelAcc_history.csv'):
    os.makedirs('./outputs', exist_ok=True)
    file_exists = os.path.isfile(log_file)
    with open(log_file, mode='a', newline='') as file:
        writer = csv.writer(file)
        if not file_exists:
            writer.writerow(['Timestamp', 'Model_Name', 'Epochs', 'Batch_Size', 
                             'Learning_Rate', 'Train_Accuracy', 'Val_Accuracy', 'Notes'])
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        writer.writerow([timestamp, model_name, epochs, batch_size, 
                         learning_rate, f"{train_acc:.4f}", f"{val_acc:.4f}", notes])

### Model Train Function

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10):
    # Check if a GPU is available, otherwise use CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")
    model.to(device)
    
    best_val_acc = 0.0

    for epoch in range(num_epochs):
        # --- 1. TRAINING PHASE ---
        model.train() # Tells layers like Dropout to turn ON
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Step 1: Clear old gradients
            optimizer.zero_grad()
            
            # Step 2: Forward pass (guess the class)
            outputs = model(images)
            
            # Step 3: Calculate the error
            loss = criterion(outputs, labels)
            
            # Step 4: Backward pass (calculate the adjustments)
            loss.backward()
            
            # Step 5: Update the weights
            optimizer.step()
            
            # Track metrics
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
        train_acc = correct_train / total_train
        
        # --- 2. VALIDATION PHASE ---
        model.eval() # Tells layers like Dropout to turn OFF (we want full power for testing)
        correct_val = 0
        total_val = 0
        val_loss = 0.0
        
        # Checking accuracy with validation data
        with torch.no_grad(): # Don't calculate gradients (saves memory and speeds up testing)
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()
                
        val_acc = correct_val / total_val
        
        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {running_loss/len(train_loader):.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {val_acc:.4f}")
        
        # Save the best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            os.makedirs('./outputs', exist_ok=True)
            torch.save(model.state_dict(), './outputs/baseline_model_v1.pth')
            
    return train_acc, best_val_acc

### Loading the Data

In [4]:
from datacleaner import build_training_dataloaders

train_loader, val_loader = build_training_dataloaders(
    csv_path='./raw_data/train_labels_strid.csv', 
    img_dir='./raw_data/train', 
    batch_size=32
)

### Execution

In [7]:
# 1. Bring in our model
from model import SimpleCNN
model = SimpleCNN(num_classes=8)

# 2. Define hyper-parameters
LEARNING_RATE = 0.001
EPOCHS = 5 # Keep it small just to test that it runs without crashing
BATCH_SIZE = 32

# 3. Define Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 4. RUN IT!
print("Starting training...")
final_train_acc, final_val_acc = train_model(
    model, train_loader, val_loader, criterion, optimizer, num_epochs=EPOCHS
)

# 5. Log the experiment
log_experiment(
    model_name="SimpleCNN_Baseline",
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    train_acc=final_train_acc,
    val_acc=final_val_acc,
    notes="Baseline Model V.1."
)

Starting training...
Training on device: cpu


KeyboardInterrupt: 